# Group images

Find correlated groups in dataset.

Algorithm:
1. Read names from CVS file
2. Remove duplicates
3. Group by prefix similarity
4. Calculate image embeddings and image similarities
5. Group by avg image similarities in groups
6. Save groups' ID

In [23]:
import os
import json
import uuid
import random
import sys

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

sys.path.append("..")
from utils import group_images, data_reading


%load_ext autoreload

%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Read names from CVS file. Remove duplicates

In [24]:
# BASE_PATH = "/mnt/pg-fs01/Current/DB/ObjectDetectionV3"
BASE_PATH = "/home/klymy/data/ObjectDetectionV3_2"
BASE_ANNO_PATH = os.path.join(BASE_PATH, "annotation")

df = pd.read_csv(os.path.join(BASE_PATH, "images.csv"))
print("'images' table shape", df.shape)

names = df["name"].values
names = np.unique(names)
print("number of unique images", len(names))

df.head()

'images' table shape (2065, 9)
number of unique images 2065


,id,name,width,height,batch,folder,task_id,job_id,job_url
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam2_H255_discaunt_29012024_2024-02-06-11h11m0...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182
1,b74d0fb1-32e7-4629-8fad-c1a606cb0fb3,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182
2,1a2a73ed-562b-4f79-8374-59eef50bea63,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182
3,e2acf72f-9e57-4f7a-a0ee-89aed453dd32,Cam2_H255_discaunt_29012024_2024-02-06-11h11m1...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182
4,29a3b2e9-5d65-4441-9588-42dea2bc372f,Cam2_H255_discaunt_29012024_2024-02-06-11h11m2...,640,480,annotation einav,einav1,653525,832182,https://app.cvat.ai/tasks/653525/jobs/832182


In [25]:
# create holdout set
df_holdout = df[
    df["name"].str.lower().str.contains("menivim") |\
    df["name"].str.lower().str.contains("commassist_cam2") |\
    df["name"].str.lower().str.contains("h240_cam2_openspace_10fps") |\
    df["name"].isin([
        "scene00051.png", "scene00701.png", "scene04551.png",
        "frame2024-03-03-21h21m07s067.png",
        "H265_Cam2_test_20fps_hallway_20211202_174003.avi_snapshot_00.00.873.bmp"
    ])
]

df_holdout.shape

holdout_names = df_holdout["name"].values

len(holdout_names)

37

In [26]:
names = np.array(list(set(names).difference(holdout_names)))
names.shape

(2028,)

## Group by prefix similarity

### Build prefix hierarchy

In [27]:
root = group_images.build_tree(names)
group_images.print_tree(root)

'': 2028
│   ├───'2017': 31
│   │   ├───'2017_06': 26
│   │   │   ├───'2017_06_21': 25
│   │   │   │   └───'2017_06_21_DL': 25
│   │   │   │       └───'2017_06_21_DL_SL': 25
│   │   │   │           └───'2017_06_21_DL_SL_Farnel02': 25
│   │   │   │               └───'2017_06_21_DL_SL_Farnel02_H270': 25
│   │   │   │                   └───'2017_06_21_DL_SL_Farnel02_H270_15m': 25
│   │   │   │                       ├───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750': 3
│   │   │   │                       │   └───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750_f': 3
│   │   │   │                       │       ├───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750_f_00312': 1
│   │   │   │                       │       │   └───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750_f_00312_02122-11': 1
│   │   │   │                       │       │       └───'2017_06_21_DL_SL_Farnel02_H270_15m_1495096750_f_00312_02122-11_1985.jpg': 1
│   │   │   │                       │       └───'2017_06_21_DL_SL_Farnel

### Split hierarchy on groups

In [28]:
groups = {}
group_images.split_by_groups(root, groups)

print("number of groups:", len(groups))
print("is all images represented in groups:", 
      sum([len(group) for group in groups.values()]) == names.shape[0])

number of groups: 195
is all images represented in groups: True


## Calculate image embeddings and image similarities

In [29]:
embedder = group_images.Embedder()


embeddings = []
for i in tqdm(names, desc="Extracting embeddings"):
    img = group_images.img_by_name(i, df, BASE_ANNO_PATH, "images")
    embeddings.append(embedder.extract_embedding_from_img(Image.fromarray(img)))


sim_scores = np.zeros((len(embeddings), len(embeddings)))
for i in tqdm(range(len(embeddings)), desc="Computing similarity scores"):
    for j in range(i, len(embeddings)):
        similarity = embedder.compute_scores(embeddings[i], embeddings[j])[0]
        sim_scores[i, j] = similarity
        sim_scores[j, i] = similarity


with open(os.path.join(BASE_PATH, "similarity_scores.json"), "w") as f:
    json.dump({"names": names.tolist(), "score_mtx": sim_scores.tolist()}, f)
with open(os.path.join(BASE_PATH, "embeddings.json"), "w") as f:
    json.dump({i: j.cpu().tolist() for i, j in zip(names, embeddings) }, f)

Device: cuda


/home/klymy/.pyenv/versions/3.10.13/envs/pgpytrain/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/klymy/.pyenv/versions/3.10.13/envs/pgpytrain/lib/python3.10/site-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(
Computing similarity scores: 100%|██████████| 2028/2028 [01:03<00:00, 31.91it/s] 


## Group by avg image similarities in groups

In [30]:
avg_inner_group_similarities, avg_outer_goup_similarities, max_outer_group_similarities = \
    group_images.compute_group_similarities(names, groups, list(groups.keys()), sim_scores)

Computing outer group similarities:   0%|          | 0/194 [00:00<?, ?it/s]

Computing outer group similarities: 100%|██████████| 194/194 [00:10<00:00, 17.73it/s]


In [31]:
# 0.55 >= good result
# 0.6 - "golden" middle
# 65 - not so good
th = 0.65
conn_mtx = (avg_outer_goup_similarities >= th).astype(int)
connected_components = group_images.find_connected_components(conn_mtx)

print(len(connected_components))

connected_components = sorted(connected_components, key=lambda x: len(x), reverse=True)

171


### Results visualization

In [32]:
# for name in np.array(list(groups.keys()))[connected_components[0]]:
#     group_images.draw_group(df, BASE_ANNO_PATH, groups[name], name, "images")

In [33]:
# for name in np.array(list(groups.keys()))[connected_components[7]]:
#     group_images.draw_group(df, BASE_ANNO_PATH, groups[name], name, "images")

In [34]:
# for name in np.array(list(groups.keys()))[connected_components[5]]:
#     group_images.draw_group(df, BASE_ANNO_PATH, groups[name], name, "images")

### Regroup images

In [35]:
group_names = list(groups.keys())

joined_groups = []
for i in connected_components:
    tmp_group = []
    for j in i:
        tmp_group += groups[group_names[j]]
    joined_groups.append(tmp_group)

sum([len(i) for i in joined_groups]) == len(names)

True

In [43]:
group_ids = [data_reading.gen_uuid() for _ in joined_groups]

df["group_id"] = [""] * df.shape[0]

for i, group in enumerate(joined_groups):
    df.loc[df["name"].isin(group), "group_id"] = group_ids[i]

df.loc[df["name"].isin(holdout_names), "group_id"] = "holdout"

print(df["group_id"].nunique() == len(group_ids) + 1)

df.to_csv(os.path.join(BASE_PATH, f"images_grouped_th_{th}_holdout.csv"), index=False)

True
